# 02 — Extract ESD index from EDIT MLRA pages

This notebook takes the MLRA catalog produced by the previous notebook and builds the next database layer:

```text
MLRA catalog
    ↓
MLRA page
    ↓
Ecological site index
```

Target output:

| database_year | mlra_symbol | mlra_name | ecoclassid | site_name | site_url | accessed_date |
|---|---|---|---|---|---|---|

This notebook does **not** parse full ESD pages yet. It only discovers site-level records and checks whether the extracted count matches the catalog-reported ecological-site count.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm


## Configuration

Set `DATABASE_YEAR` to the annual snapshot year. The raw HTML cache is versioned by year so later STEP builds can reproduce the exact source state used for an ESD database snapshot.


In [ ]:
DATABASE_YEAR = datetime.now().year
CATALOG_CSV = Path('/mnt/data/mlra_catalog.csv')  # change if running locally

BASE_DIR = Path(f'esd_build_{DATABASE_YEAR}')
RAW_MLRA_DIR = BASE_DIR / 'raw_html' / 'mlra_pages'
TABLE_DIR = BASE_DIR / 'tables'
META_DIR = BASE_DIR / 'metadata'

for d in [RAW_MLRA_DIR, TABLE_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REQUEST_DELAY_SECONDS = 0.75
TIMEOUT_SECONDS = 30
USER_AGENT = 'STEP-ESD-index-builder/0.1 (research; polite rate-limited)'

print('Database year:', DATABASE_YEAR)
print('Output directory:', BASE_DIR.resolve())


## Load and validate MLRA catalog

The parser requires these fields from the previous notebook:

- `mlra_symbol`
- `mlra_name`
- `mlra_url`
- `ecological_site_count`


In [ ]:
mlra_catalog = pd.read_csv(CATALOG_CSV, dtype={'mlra_symbol': str, 'edit_unit_id': str})
required = {'mlra_symbol', 'mlra_name', 'mlra_url', 'ecological_site_count'}
missing = required - set(mlra_catalog.columns)
if missing:
    raise ValueError(f'MLRA catalog missing required columns: {missing}')

mlra_catalog['ecological_site_count'] = pd.to_numeric(mlra_catalog['ecological_site_count'], errors='coerce').fillna(0).astype(int)
mlra_catalog = mlra_catalog.sort_values('mlra_symbol').reset_index(drop=True)

print('MLRAs:', len(mlra_catalog))
print('Catalog site count:', mlra_catalog['ecological_site_count'].sum())
mlra_catalog.head()


## Fetching with local cache

Each MLRA page is saved once under `raw_html/mlra_pages/`. Re-running the notebook will use the cached HTML unless `force=True` is passed.


In [ ]:
session = requests.Session()
session.headers.update({'User-Agent': USER_AGENT})

def safe_filename(mlra_symbol: str) -> str:
    return re.sub(r'[^A-Za-z0-9_-]+', '_', mlra_symbol)

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8', errors='replace')).hexdigest()

def fetch_mlra_html(mlra_symbol: str, mlra_url: str, force: bool = False) -> dict:
    out_path = RAW_MLRA_DIR / f'{safe_filename(mlra_symbol)}.html'
    meta_path = RAW_MLRA_DIR / f'{safe_filename(mlra_symbol)}.json'

    if out_path.exists() and not force:
        html = out_path.read_text(encoding='utf-8', errors='replace')
        return {
            'mlra_symbol': mlra_symbol,
            'mlra_url': mlra_url,
            'html': html,
            'from_cache': True,
            'status_code': None,
            'html_path': str(out_path),
            'html_hash': sha256_text(html),
        }

    r = session.get(mlra_url, timeout=TIMEOUT_SECONDS)
    r.raise_for_status()
    html = r.text
    out_path.write_text(html, encoding='utf-8')

    meta = {
        'mlra_symbol': mlra_symbol,
        'mlra_url': mlra_url,
        'status_code': r.status_code,
        'fetched_utc': datetime.now(timezone.utc).isoformat(),
        'html_hash': sha256_text(html),
        'html_path': str(out_path),
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding='utf-8')
    time.sleep(REQUEST_DELAY_SECONDS)

    return {
        'mlra_symbol': mlra_symbol,
        'mlra_url': mlra_url,
        'html': html,
        'from_cache': False,
        'status_code': r.status_code,
        'html_path': str(out_path),
        'html_hash': meta['html_hash'],
    }


## Parse ecological-site records from one MLRA page

The parser is intentionally conservative. It looks for links matching:

```text
/catalogs/esd/{MLRA}/{ECOCLASSID}
```

Then it extracts the nearest visible title/name from the surrounding list item. This should be more stable than relying on one specific CSS class.


In [ ]:
SITE_HREF_RE = re.compile(r'/catalogs/esd/([A-Za-z0-9]+)/(R[A-Za-z0-9]+)')

def clean_text(x: str) -> str:
    return re.sub(r'\s+', ' ', x or '').strip()

def extract_site_name(anchor, ecoclassid: str) -> str | None:
    # Prefer EDIT list markup when available.
    container = anchor.find_parent(['li', 'tr', 'div'])
    candidates = []

    if container is not None:
        for cls in ['list-item-name', 'site-name', 'site-title']:
            el = container.find(class_=cls)
            if el:
                candidates.append(clean_text(el.get_text(' ')))
        # Common list-item block: symbol + name both inside link.
        txt = clean_text(container.get_text(' '))
        candidates.append(txt)

    candidates.append(clean_text(anchor.get_text(' ')))

    # Clean candidates: remove ecoclassid and short noise.
    cleaned = []
    for c in candidates:
        c = c.replace(ecoclassid, '').strip(' -:|')
        c = re.sub(r'\bAdd to briefcase\b.*$', '', c, flags=re.I).strip()
        c = re.sub(r'\bQuick view\b.*$', '', c, flags=re.I).strip()
        c = re.sub(r'\bDownload data\b.*$', '', c, flags=re.I).strip()
        if c and len(c) > 2 and c.lower() != ecoclassid.lower():
            cleaned.append(c)

    # Select the shortest informative candidate to avoid whole-card text.
    if cleaned:
        return sorted(cleaned, key=len)[0]
    return None

def parse_mlra_sites(html: str, mlra_symbol: str, mlra_name: str | None = None) -> pd.DataFrame:
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    seen = set()

    for a in soup.find_all('a', href=True):
        href = a['href']
        m = SITE_HREF_RE.search(href)
        if not m:
            continue
        mlra_from_href, ecoclassid = m.groups()
        if mlra_from_href.upper() != mlra_symbol.upper():
            continue
        if ecoclassid in seen:
            continue
        seen.add(ecoclassid)

        if href.startswith('http'):
            site_url = href.replace('http://', 'https://')
        else:
            site_url = f'https://edit.sc.egov.usda.gov{href}'

        rows.append({
            'database_year': DATABASE_YEAR,
            'mlra_symbol': mlra_symbol,
            'mlra_name': mlra_name,
            'ecoclassid': ecoclassid,
            'site_name': extract_site_name(a, ecoclassid),
            'site_url': site_url,
            'accessed_date': datetime.now(timezone.utc).date().isoformat(),
        })

    return pd.DataFrame(rows)


## Smoke test: one MLRA

Use a known MLRA first. `011X` is useful because we already inspected `R011XY001ID` manually.


In [ ]:
test_mlra = '011X'
test_row = mlra_catalog.loc[mlra_catalog['mlra_symbol'] == test_mlra].iloc[0]
result = fetch_mlra_html(test_row.mlra_symbol, test_row.mlra_url)
test_sites = parse_mlra_sites(result['html'], test_row.mlra_symbol, test_row.mlra_name)

print('Expected site count:', int(test_row.ecological_site_count))
print('Extracted site count:', len(test_sites))
test_sites.head(10)


If `Extracted site count` does not match `Expected site count`, inspect the cached HTML and adjust the parser before running the full catalog.


In [ ]:
# Optional diagnostic for the smoke test
if len(test_sites) != int(test_row.ecological_site_count):
    print('WARNING: count mismatch for smoke test')
    print('Cached HTML:', result['html_path'])
else:
    print('Smoke test passed.')


## Full extraction: MLRA → ESD index

This loops over every MLRA in the catalog. It writes intermediate files so an interrupted run can be resumed.


In [ ]:
all_site_frames = []
fetch_records = []
errors = []

for row in tqdm(mlra_catalog.itertuples(index=False), total=len(mlra_catalog)):
    try:
        fetched = fetch_mlra_html(row.mlra_symbol, row.mlra_url)
        sites = parse_mlra_sites(fetched['html'], row.mlra_symbol, row.mlra_name)
        sites['expected_site_count'] = int(row.ecological_site_count)
        sites['mlra_html_hash'] = fetched['html_hash']
        sites['mlra_html_path'] = fetched['html_path']
        all_site_frames.append(sites)

        fetch_records.append({
            'mlra_symbol': row.mlra_symbol,
            'mlra_name': row.mlra_name,
            'mlra_url': row.mlra_url,
            'expected_site_count': int(row.ecological_site_count),
            'extracted_site_count': len(sites),
            'from_cache': fetched['from_cache'],
            'status_code': fetched['status_code'],
            'html_hash': fetched['html_hash'],
            'html_path': fetched['html_path'],
        })
    except Exception as e:
        errors.append({
            'mlra_symbol': row.mlra_symbol,
            'mlra_name': row.mlra_name,
            'mlra_url': row.mlra_url,
            'error': repr(e),
        })

esd_index = pd.concat(all_site_frames, ignore_index=True) if all_site_frames else pd.DataFrame()
fetch_log = pd.DataFrame(fetch_records)
error_log = pd.DataFrame(errors)

print('Extracted ESD records:', len(esd_index))
print('Catalog expected records:', int(mlra_catalog['ecological_site_count'].sum()))
print('MLRA errors:', len(error_log))


## Completeness checks

The main validation criterion is whether extracted site counts match the catalog counts per MLRA and in total.


In [ ]:
count_check = (
    mlra_catalog[['mlra_symbol', 'mlra_name', 'ecological_site_count']]
    .rename(columns={'ecological_site_count': 'expected_site_count'})
    .merge(
        esd_index.groupby('mlra_symbol').size().rename('extracted_site_count').reset_index(),
        on='mlra_symbol',
        how='left'
    )
)
count_check['extracted_site_count'] = count_check['extracted_site_count'].fillna(0).astype(int)
count_check['count_difference'] = count_check['extracted_site_count'] - count_check['expected_site_count']
count_check['matches_catalog'] = count_check['count_difference'].eq(0)

print('Matching MLRAs:', int(count_check['matches_catalog'].sum()), '/', len(count_check))
print('Total expected:', int(count_check['expected_site_count'].sum()))
print('Total extracted:', int(count_check['extracted_site_count'].sum()))

count_check.loc[~count_check['matches_catalog']].head(20)


In [ ]:
# Duplicate diagnostics
if not esd_index.empty:
    dupes = esd_index[esd_index.duplicated(['mlra_symbol', 'ecoclassid'], keep=False)].sort_values(['mlra_symbol', 'ecoclassid'])
    print('Duplicate MLRA/ecoclassid rows:', len(dupes))
    display(dupes.head(20))


## Save outputs

Outputs are written as CSV and Parquet when possible. CSV is the exchange format; Parquet is faster and preserves types better.


In [ ]:
esd_index_csv = TABLE_DIR / 'esd_index.csv'
count_check_csv = TABLE_DIR / 'esd_index_count_check.csv'
fetch_log_csv = META_DIR / 'mlra_fetch_log.csv'
error_log_csv = META_DIR / 'mlra_error_log.csv'

esd_index.to_csv(esd_index_csv, index=False)
count_check.to_csv(count_check_csv, index=False)
fetch_log.to_csv(fetch_log_csv, index=False)
error_log.to_csv(error_log_csv, index=False)

try:
    esd_index.to_parquet(TABLE_DIR / 'esd_index.parquet', index=False)
    count_check.to_parquet(TABLE_DIR / 'esd_index_count_check.parquet', index=False)
except Exception as e:
    print('Parquet export skipped:', repr(e))

run_metadata = {
    'database_year': DATABASE_YEAR,
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'mlra_count': int(len(mlra_catalog)),
    'catalog_expected_site_count': int(mlra_catalog['ecological_site_count'].sum()),
    'extracted_site_count': int(len(esd_index)),
    'matching_mlra_count': int(count_check['matches_catalog'].sum()),
    'error_count': int(len(error_log)),
    'request_delay_seconds': REQUEST_DELAY_SECONDS,
    'timeout_seconds': TIMEOUT_SECONDS,
    'parser': '02_extract_esd_index.ipynb',
}
(META_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')

print('Saved:')
print(' ', esd_index_csv)
print(' ', count_check_csv)
print(' ', fetch_log_csv)
print(' ', error_log_csv)
print(' ', META_DIR / 'run_metadata.json')


## Next notebook

After this notebook passes the completeness check, the next notebook should parse full ESD pages:

```text
03_extract_esd_detail_pages.ipynb
```

Input:

```text
esd_index.csv
```

First output target:

```text
raw_html/esd_pages/{ecoclassid}.html
parsed_json/{ecoclassid}.json
```

Do not normalize state-transition edges until raw section/table extraction is stable across multiple MLRAs.
